# Flux foundation photometry head

This notebook visualizes the learned photometry head, not the per-scene scarlet optimizer.

Inputs:

1. CenterNet detections in the VIS frame.
2. V8 frozen foundation features for the full 10-band tile.
3. Astrometry-head corrected source positions.
4. PSFField-rendered per-band PSFs.
5. A trained `FoundationScarletPhotometryHead` checkpoint.

The head predicts morphology refinements from V8 features. Fluxes are solved through the differentiable renderer, and the metric is residual chi-square on Euclid-native VIS/Y/J/H scenes.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 120
plt.rcParams['image.origin'] = 'lower'


def find_repo_root(start: Path = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for cand in [start, *start.parents]:
        if (cand / 'models').exists() and (cand / 'data').exists():
            return cand
    return start

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'models'))

from models.photometry import PSFFieldPhotometryPipeline
from models.photometry.foundation_head import photometry_head_loss
from models.photometry.scarlet_like import build_neighbor_groups, make_positive_morphology_templates
from models.photometry.train_foundation_photometry_head import (
    EUCLID_BANDS,
    apply_astrometry_correction,
    detect_vis_positions,
    filter_positions_in_all_bands,
    load_centernet_detector,
    load_encoder_and_astrometry,
    load_euclid_native_arrays,
    make_head_from_config,
    project_vis_positions_to_euclid_bands,
)
from astrometry2.dataset import discover_tile_pairs
from astrometry2.train_latent_position import load_tile_data

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'ROOT   = {ROOT}')
print(f'DEVICE = {DEVICE}')


In [ ]:
RUBIN_DIR = ROOT / 'data' / 'rubin_tiles_all'
EUCLID_DIR = ROOT / 'data' / 'euclid_tiles_all'
FOUNDATION_CKPT = ROOT / 'models' / 'checkpoints' / 'jaisp_v10_warmstart' / 'checkpoint_best.pt'
CENTERNET_CKPT = ROOT / 'checkpoints' / 'centernet_v10_uncertain_synth_r2' / 'centernet_best.pt'
ASTROMETRY_CKPT = ROOT / 'models' / 'checkpoints' / 'latent_position_v10_no_psf' / 'best.pt'
PSF_CKPT = ROOT / 'models' / 'checkpoints' / 'psf_field_v3.pt'
HEAD_CKPT = ROOT / 'models' / 'checkpoints' / 'photometry_foundation_200_emppsf' / 'checkpoint_best.pt'
SMOKE_HEAD_CKPT = ROOT / 'models' / 'checkpoints' / 'photometry_foundation_smoke' / 'checkpoint_best.pt'
OUT_DIR = ROOT / 'io'

if not HEAD_CKPT.exists() and SMOKE_HEAD_CKPT.exists():
    print(f'Using smoke checkpoint because full checkpoint is missing: {SMOKE_HEAD_CKPT.relative_to(ROOT)}')
    HEAD_CKPT = SMOKE_HEAD_CKPT

MAX_SOURCES = 12
MAX_SOURCES_PER_STEP = 10
MORPH_SIZE = 31
MARGIN = 38
GROUP_RADIUS = 9.0
SCENE_MIN = 49
SCENE_MAX = 71
SUB_GRID = 1
DETECTOR_CONF = 0.30

for label, p in [
    ('Foundation', FOUNDATION_CKPT),
    ('CenterNet', CENTERNET_CKPT),
    ('Astrometry', ASTROMETRY_CKPT),
    ('PSFField', PSF_CKPT),
    ('Phot head', HEAD_CKPT),
]:
    print(f'{label:12s}: {p.relative_to(ROOT) if p.exists() else p}  {"OK" if p.exists() else "MISSING"}')


In [ ]:
assert HEAD_CKPT.exists(), 'Train first: python models/photometry/train_foundation_photometry_head.py --output-dir models/checkpoints/photometry_foundation_200_emppsf'

pairs = discover_tile_pairs(str(RUBIN_DIR), str(EUCLID_DIR))
tile_id, rubin_path, euclid_path = pairs[0]
print(f'Using tile: {tile_id}')

frozen_encoder, astro_head = load_encoder_and_astrometry(str(FOUNDATION_CKPT), str(ASTROMETRY_CKPT), DEVICE)
detector = load_centernet_detector(str(FOUNDATION_CKPT), str(CENTERNET_CKPT), DEVICE)
psf_pipe = PSFFieldPhotometryPipeline.from_checkpoint(
    PSF_CKPT,
    band_names=EUCLID_BANDS,
    stamp_size=MORPH_SIZE,
    sub_grid=SUB_GRID,
    bg_inner_radius=10.0,
    bg_outer_radius=14.0,
    device=DEVICE,
)

ckpt = torch.load(HEAD_CKPT, map_location='cpu', weights_only=False)
head = make_head_from_config(ckpt['config']).to(DEVICE)
head.load_state_dict(ckpt['head_state_dict'])
head.eval()
print(f'Loaded photometry head epoch={ckpt.get("epoch", "?")} from {HEAD_CKPT.relative_to(ROOT)}')


In [ ]:
context_images, context_rms, vis_hw, vis_wcs = load_tile_data(rubin_path, euclid_path, DEVICE)
enc_out = frozen_encoder.encode_tile(context_images, context_rms)
euclid_tile_cpu, euclid_rms_cpu, band_wcs = load_euclid_native_arrays(euclid_path)
euclid_tile = euclid_tile_cpu.to(DEVICE)
euclid_rms = euclid_rms_cpu.to(DEVICE)

det_xy = detect_vis_positions(
    tile_id,
    rubin_path,
    euclid_path,
    detector,
    DEVICE,
    conf_threshold=DETECTOR_CONF,
    max_sources=MAX_SOURCES,
    margin=MARGIN,
    use_classical=False,
)
corrected_vis = apply_astrometry_correction(astro_head, enc_out, det_xy, vis_wcs, DEVICE)
positions_by_band = project_vis_positions_to_euclid_bands(corrected_vis, vis_wcs, band_wcs, DEVICE)
keep = filter_positions_in_all_bands(corrected_vis, positions_by_band, tuple(euclid_tile.shape[-2:]), MARGIN)
corrected_vis = corrected_vis[keep][:MAX_SOURCES_PER_STEP]
positions_by_band = positions_by_band[keep][:MAX_SOURCES_PER_STEP]
print(f'CenterNet detections: {len(det_xy)}')
print(f'Astrometry-corrected sources used: {corrected_vis.shape[0]}')


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
vis = euclid_tile_cpu[0].numpy()
vmin, vmax = np.nanpercentile(vis, [2, 99.5])
ax.imshow(vis, cmap='gray', vmin=vmin, vmax=vmax)
ax.scatter(det_xy[:, 0], det_xy[:, 1], s=40, facecolors='none', edgecolors='tab:orange', label='CenterNet')
ax.scatter(corrected_vis[:, 0].cpu(), corrected_vis[:, 1].cpu(), s=18, c='tab:cyan', label='astrometry corrected')
ax.set_title('VIS detections and corrected positions')
ax.set_xticks([]); ax.set_yticks([])
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
morph_pack = make_positive_morphology_templates(
    euclid_tile[0],
    corrected_vis,
    stamp_size=MORPH_SIZE,
    bg_inner_radius=11.0,
    bg_outer_radius=15.0,
    smooth_sigma=0.6,
)
init_morph = morph_pack['templates']
groups = build_neighbor_groups(corrected_vis.detach().cpu(), radius_px=GROUP_RADIUS)
with torch.no_grad():
    psfs = psf_pipe.render_psfs(positions_by_band, tile_hw=euclid_tile.shape[-2:])
print('Morphology templates:', tuple(init_morph.shape))
print('PSFs:', tuple(psfs.shape))
print('Groups:', [len(g) for g in groups])


In [ ]:
with torch.no_grad():
    learned = photometry_head_loss(
        head,
        bottleneck=enc_out['bottleneck'],
        vis_stem=enc_out['vis_stem'],
        source_positions_vis=corrected_vis,
        init_morphology=init_morph,
        tile=euclid_tile,
        rms=euclid_rms,
        positions_px=positions_by_band,
        psfs=psfs,
        fused_hw=enc_out['fused_hw'],
        vis_hw=enc_out['vis_hw'],
        groups=groups,
        min_scene_size=SCENE_MIN,
        max_scene_size=SCENE_MAX,
        return_scenes=True,
    )

psf_out = psf_pipe.run(
    euclid_tile,
    euclid_rms,
    positions_by_band,
    return_psfs=True,
    return_stamps=True,
)

learned_chi2 = learned['chi2_dof'].detach().cpu().numpy()
psf_chi2 = psf_out['chi2_dof'].detach().cpu().numpy()
print('Median chi2/dof, same astrometry-corrected catalog')
for b, name in enumerate(EUCLID_BANDS):
    print(f'{name:10s}  PSF={np.nanmedian(psf_chi2[:, b]):8.3f}  learned={np.nanmedian(learned_chi2[:, b]):8.3f}')


In [ ]:
def savefig(path):
    plt.savefig(path, dpi=200, bbox_inches='tight')
    print(f'Saved {Path(path).relative_to(ROOT)}')

fig, axes = plt.subplots(1, len(EUCLID_BANDS), figsize=(4 * len(EUCLID_BANDS), 3.5))
for b, ax in enumerate(axes):
    x = np.clip(psf_chi2[:, b], 1e-3, None)
    y = np.clip(learned_chi2[:, b], 1e-3, None)
    ax.scatter(x, y, s=28, alpha=0.75)
    lim = max(2.0, np.nanpercentile(np.concatenate([x, y]), 95))
    ax.plot([1e-3, lim], [1e-3, lim], color='0.4', lw=1)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(1e-2, lim); ax.set_ylim(1e-2, lim)
    ax.set_title(EUCLID_BANDS[b])
    ax.set_xlabel('PSF-only chi2/dof')
    if b == 0:
        ax.set_ylabel('learned-head chi2/dof')
    ax.grid(alpha=0.25)
fig.suptitle('V8 photometry head residual comparison', y=1.04)
plt.tight_layout()
savefig(OUT_DIR / 'foundation_photometry_head_chi2_compare.png')
plt.show()


In [ ]:
def scene_band_chi2(scene, band=0):
    return float(scene.chi2_dof[band])


def plot_scene_gallery(scene_results, band=0, mode='good', n_show=6, save_path=None):
    scenes = list(scene_results)
    scenes = sorted(scenes, key=lambda s: scene_band_chi2(s, band), reverse=(mode == 'worst'))[:n_show]
    fig, axes = plt.subplots(len(scenes), 3, figsize=(8, 2.5 * len(scenes)))
    if len(scenes) == 1:
        axes = axes[None, :]
    for row, scene in enumerate(scenes):
        data = scene.data[band].numpy()
        bg = float(scene.background[band])
        model = scene.model[band].numpy()
        resid = scene.resid[band].numpy()
        data_sub = data - bg
        model_sub = model - bg
        vmax = np.nanpercentile(np.abs(data_sub), 99)
        rv = np.nanpercentile(np.abs(resid), 98)
        panels = [
            (data_sub, 'data-bg', 'magma', -vmax, vmax),
            (model_sub, 'learned model', 'magma', 0, np.nanpercentile(model_sub, 99)),
            (resid, 'residual', 'coolwarm', -rv, rv),
        ]
        for col, (arr, title, cmap, vmin, vmax_i) in enumerate(panels):
            ax = axes[row, col]
            ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax_i)
            ax.set_xticks([]); ax.set_yticks([])
            if row == 0:
                ax.set_title(title)
        axes[row, 0].set_ylabel(f'g{scene.group_id} n={len(scene.indices)}\nchi2={scene_band_chi2(scene, band):.2f}', fontsize=8)
    fig.suptitle(f'Foundation photometry head - {EUCLID_BANDS[band]} ({mode})', y=1.01)
    plt.tight_layout()
    if save_path:
        savefig(save_path)
    plt.show()

plot_scene_gallery(
    learned['scene_results'],
    band=0,
    mode='good',
    save_path=OUT_DIR / 'foundation_photometry_head_good_gallery.png',
)
plot_scene_gallery(
    learned['scene_results'],
    band=0,
    mode='worst',
    save_path=OUT_DIR / 'foundation_photometry_head_worst_gallery.png',
)
